# Notebook 4 : Station spatiale internationale 🛰️

In [ ]:
# Décommenter la ligne suivante pour installer les dépendances
# %pip install folium jupyter_bokeh nbconvert panel watchfiles

In [ ]:
import folium
import pandas as pd
import panel as pn
import requests

pn.extension()

La [station spatiale internationale](https://fr.wikipedia.org/wiki/Station_spatiale_internationale) (*ISS*) est en orbite autour de notre planète à une altitude légérement supérieure à 400 km. Cette station effectue une quinzaine de révolutions par jour et nous pouvons suivre [sa position en temps réel](https://spotthestation.nasa.gov/tracking_map.cfm) grâce à différents outils dont l'API de [Where the ISS at](https://wheretheiss.at/). Nous proposons dans la suite de mettre en place une application web de visualisation des données de l'ISS.

## Préliminaires

1. Utiliser la fonction `get` de `requests` pour récupérer les informations sur la position courante de l'ISS à partir de l'API de [Where the ISS at](https://wheretheiss.at/) à l'adresse suivante :<br/>[https://api.wheretheiss.at/v1/satellites/25544](https://api.wheretheiss.at/v1/satellites/25544)<br/>Comprendre en particulier à quoi correspond la valeur `timestamp`.

In [ ]:
r = requests.get("https://api.wheretheiss.at/v1/satellites/25544")

if r.status_code == 200:
    # La requête est valide
    iss_json = r.json()
    print(iss_json)

    # Le timestamp est l'instant de la mesure au format de temps POSIX.
    # https://fr.wikipedia.org/wiki/Heure_Unix
    # Il s'agit d'un nombre de secondes depuis l'origine du temps selon la norme
    # POSIX qui est fixée au 1er janvier 1970. Cet instant est appelé l'epoch.
    print(f"Timestamp: {iss_json['timestamp']}")
else:
    # La requête a échoué
    print(f"Erreur de la requête (code {r.status_code})")

2. Écrire une fonction `get_iss_position` sans argument qui retourne les informations sur la position courante de l'ISS sous la forme d'un dictionnaire Python ou `None` si la requête échoue. Expliquer pourquoi cette fonction ne doit pas être décorée avec `pn.cache`.

In [ ]:
# La fonction get_iss_position ne retourne pas le même résultat à chaque appel
# car l'ISS est en mouvement. Ce résultat ne doit pas être mis en cache car les
# valeurs précédentes ne peuvent pas être recyclées lors de nouveaux appels de
# la fonction.

def get_iss_position():
    r = requests.get("https://api.wheretheiss.at/v1/satellites/25544")
    return r.json() if r.status_code == 200 else None

## Folium

Le module [Folium](https://python-visualization.github.io/folium/latest/) permet de visualiser des données géostatistiques dans une application web à partir de la bibliothèque JavaScript Leaflet.

3. Afficher une carte vierge du monde avec la fonction `Map` de `folium`.

In [ ]:
folium.Map()

4. Récupérer la position de l'ISS dans un objet `iss_0` avec `get_iss_position` et afficher un marqueur (voir [la documentation](https://python-visualization.github.io/folium/latest/getting_started.html#Adding-markers)) sur la mappemonde.

In [ ]:
# Position de l'ISS
iss_0 = get_iss_position()
if iss_0 is not None:
    position_0 = (iss_0["latitude"], iss_0["longitude"])
else:
    # Coordonnées nulles en cas d'erreur
    position_0 = (0.0, 0.0)

# Carte vierge du monde
m = folium.Map()

# Ajout d'un marqueur
folium.Marker(location=position_0).add_to(m)

m

5. Récupérer à nouveau la position de l'ISS dans un objet `iss_1` avec `get_iss_position` et afficher une ligne entre la position précédente et la nouvelle (voir [la documentation](https://python-visualization.github.io/folium/latest/getting_started.html#Vectors-such-as-lines)) sur la mappemonde.

In [ ]:
# Position de l'ISS
iss_1 = get_iss_position()
if iss_1 is not None:
    position_1 = (iss_1["latitude"], iss_1["longitude"])
else:
    # Coordonnées nulles en cas d'erreur
    position_1 = (0.0, 0.0)

# Carte vierge du monde
m = folium.Map()

# Ajout d'une ligne
folium.PolyLine([position_0, position_1]).add_to(m)

m

6. Récupérer à nouveau la position de l'ISS dans un objet `iss_2` avec `get_iss_position` et stocker les trois positions dans un DataFrame Pandas `iss_positions`.

In [ ]:
iss_2 = get_iss_position()
iss_positions = pd.DataFrame([iss_0, iss_1, iss_2])
iss_positions

7. Écrire une fonction `get_iss_map` qui prend un DataFrame tel que `iss_positions` en argument et retourne une carte Folium affichant la trajectoire de l'ISS avec une ligne et sa dernière position avec un marqueur.

In [ ]:
def get_iss_map(iss_positions):
    # Tri par timestamp croissant
    iss_positions = iss_positions.sort_values(by="timestamp")
    # Liste des coordonnées des positions
    coords = list(
        iss_positions[["latitude", "longitude"]]
        .itertuples(index=False, name=None)
    )

    # Carte vierge du monde
    m = folium.Map()
    # Affichage de la trajectoire
    folium.PolyLine(coords).add_to(m)
    # Affichage de la dernière position
    folium.Marker(
        location=coords[-1],
        # Bonus : ajoute un tooltip
        tooltip="Station spatiale internationale",
    ).add_to(m)

    return m

## Vers l'ISS et au-delà

8. Créer un *pane* `iss_df` de type `DataFrame` (voir [la documentation](https://panel.holoviz.org/reference/panes/DataFrame.html)) initialement vide et destiné à contenir un DataFrame Pandas avec trois colonnes `timestamp`, `latitude` et `longitude`.

In [ ]:
iss_df = pn.pane.DataFrame(
    # La colonne velocity est ajoutée pour la question 12
    pd.DataFrame(columns=["timestamp", "latitude", "longitude", "velocity"]),
    # Bonus : ne pas afficher la colonne index
    index=False,
)

9. Créer un widget `iss_button` de type `Button` (voir [la documentation](https://panel.holoviz.org/reference/widgets/Button.html)) et le lier à une fonction `update_position` qui sera appelée lorsque le bouton est pressé pour :

- récupérer la position courante de l'ISS avec `get_iss_position`,
- ajouter une ligne correspondante dans le DataFrame du *pane* `iss_df`,
- appeler `get_iss_map` et retourner le résultat.

In [ ]:
# Question 12
vitesse = pn.indicators.Number(
    name="Vitesse",
    value=0,
    format="{value:.2f} km/h",
    font_size="18pt",
)

def update_position(clicked):
    # Récupération de la position de l'ISS
    iss_position = get_iss_position()
    if iss_position is None:
        # Ne rien faire si la requête a échoué
        return
    
    # Le DataFrame se trouve dans l'attribut object du pane
    positions = iss_df.object

    # Extraction des données à ajouter
    new_position = {key: iss_position[key] for key in positions.columns}
    
    # Mise à jour de la vitesse (Question 12)
    vitesse.value = new_position["velocity"]

    # Mise à jour du DataFrame
    iss_df.object = pd.concat(
        [
            # La condition évite un message d'erreur pour un DataFrame vide
            positions if len(positions) > 0 else None,
            # Un DataFrame d'une ligne doit fournir un index
            pd.DataFrame(new_position, index=[0]),
        ],
        ignore_index=True,
    )

    # Retourne la carte mise à jour
    return get_iss_map(iss_df.object)

iss_button = pn.widgets.Button(name="Position ISS")
iss_map = pn.bind(update_position, iss_button)

## Application web

Nous pouvons maintenant mettre en place une application web qui pourra être démarrée avec la commande suivante (l'option `--allow-websocket-origin` n'est nécessaire que dans Onyxia) :
```{bash}
panel serve --autoreload --show --allow-websocket-origin=$(echo $VSCODE_PROXY_URI | cut -d '/' -f 3) notebooks/04_iss.ipynb
```

Les questions suivantes ont pour objet d'enrichir l'application au fur et à mesure. Il ne faut donc pas recréer une nouvelle application pour chaque question mais faire évoluer le code étape par étape. Il peut être utile d'ajouter de nouvelles cellules de code si besoin.

10. Mettre en forme une application à l'aide du modèle `FastListTemplate` (voir [la documentation](https://panel.holoviz.org/reference/templates/FastListTemplate.html)) avec le bouton `iss_button` et le *pane* `iss_df` dans la barre latérale et un *pane* Folium (voir [la documentation](https://panel.holoviz.org/reference/panes/Folium.html)) contenant la carte obtenue grâce à la fonction liée `update_position` dans la zone principale.

11. Utiliser le paramètre `sizing_mode` du *pane* Folium pour adapter sa taille à la fenêtre.

12. *(Bonus)* Adapter l'application pour afficher la vitesse de l'ISS dans un indicateur de type `Number` (voir [la documentation](https://panel.holoviz.org/reference/indicators/Number.html)).

In [ ]:
pn.template.FastListTemplate(
    title="Station spatiale internationale 🛰️",
    sidebar=[iss_button, iss_df],
    main=[
        pn.Row(
            vitesse, # Question 12
            pn.pane.plot.Folium(
                iss_map,
                sizing_mode="stretch_both", # Question 11
            ),
        )
    ],
).servable()